### Task V: Quantum Graph Neural Network (QGNN)
In task II you already worked with a classical GNN.
Describe a possibility for a QGNN circuit, which takes advantage of the graph representation of the data
Implement and draw the circuit.


In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
# 1. Define the Number of Qubits (one per particle)
n_particles = 4
dev = qml.device("default.qubit", wires=n_particles)

In [3]:
# 2. Define the Graph Structure (Edges)
# In a jet, this could be k-Nearest Neighbors based on distance
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]

In [4]:
@qml.qnode(dev)
def qgnn_circuit(inputs, weights):
    """
    inputs: Scaled physical features of 4 particles (e.g., pT, eta, phi)
    weights: Trainable variational parameters for the GNN layers
    """

    # --- ENCODING LAYER (Nodes) ---
    # Encode classical features into the rotation of each qubit
    for i in range(n_particles):
        qml.RY(inputs[i], wires=i)

    # --- INTERACTION LAYER (Edges / Message Passing) ---
    # We use the graph edges to determine where to apply entangling gates
    # This allows information to flow between "connected" particles
    for i, edge in enumerate(edges):
        qml.CNOT(wires=[edge[0], edge[1]])
        # Parametrized rotation represents the "learned" interaction strength
        qml.RZ(weights[i], wires=edge[1])
        qml.CNOT(wires=[edge[0], edge[1]])

    # --- READOUT (Quantum Pooling) ---
    # We measure only the first qubit.
    # Because of the CNOTs above, its state now depends on ALL other qubits.
    return qml.expval(qml.PauliZ(0))

In [5]:
# 3. Example Execution
# Random normalized features for 4 particles [0, pi]
node_features = np.array([0.5, 1.2, 2.1, 0.8], requires_grad=False)
# One weight per edge in the graph
trainable_weights = np.array([0.1, -0.2, 0.4, 0.9], requires_grad=True)

In [6]:
# Draw the circuit to visualize the "Graph" topology
print("QGNN Circuit Diagram:")
print(qml.draw(qgnn_circuit)(node_features, trainable_weights))

QGNN Circuit Diagram:
0: ──RY(0.50)─╭●───────────╭●──────────────────────────────────╭X──RZ(0.90)─╭X─┤  <Z>
1: ──RY(1.20)─╰X──RZ(0.10)─╰X─╭●────────────╭●─────────────────│────────────│──┤     
2: ──RY(2.10)─────────────────╰X──RZ(-0.20)─╰X─╭●───────────╭●─│────────────│──┤     
3: ──RY(0.80)──────────────────────────────────╰X──RZ(0.40)─╰X─╰●───────────╰●─┤     


In [7]:
# Run the circuit
result = qgnn_circuit(node_features, trainable_weights)
print(f"\nClassification Score (Expectation Value): {result:.4f}")


Classification Score (Expectation Value): 0.8776
